# 04b — Driver Manifestations (Morphological Box)

For each unified technology driver, determine **2–4 domain-specific manifestations** (Ausprägungen) — concrete, mutually exclusive states this driver could reach by 2035.

These manifestations form the **Zwicky Box** (morphological box) that will be combined into scenario skeletons in NB05b.

- Input: `merge_state.json` (14 unified drivers)
- Output: `morphbox_state.json` (drivers × manifestations)

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from src.llm import safe_chat_json
from src.rag import get_collection, retrieve, format_rag_chunks
from src.models.drivers import TechDriver
from src.models.morphological import DriverManifestation, MorphologicalBox
from src import prompts

with open("../data/outputs/merge_state.json") as f:
    merge_state = json.load(f)
drivers = [TechDriver(**d) for d in merge_state["unified_drivers"]]

collection = get_collection()
print(f"Loaded {len(drivers)} drivers, KB collection: {collection.count()} chunks")

In [ ]:
all_manifestations: list[DriverManifestation] = []
manifestation_map: dict[str, list[str]] = {}

for i, driver in enumerate(drivers):
    print(f"\n[{i+1}/{len(drivers)}] {driver.name[:60]}")

    chunks = retrieve(collection, f"{driver.name} {driver.description[:100]}", n=5)
    rag_text = format_rag_chunks(chunks)

    prompt = prompts.MANIFESTATION_DETERMINE.format(
        driver_name=driver.name,
        driver_description=driver.description,
        driver_origin=driver.origin.value,
        driver_confidence=driver.confidence.value,
        rag_chunks=rag_text,
    )

    result = safe_chat_json(prompt, system="You are determining technology manifestations for regulatory frequency monitoring foresight. Be specific and domain-grounded.")

    manif_ids = []
    for m in result.get("manifestations", []):
        dm = DriverManifestation(
            driver_id=driver.id,
            label=m["label"],
            description=m["description"],
            plausibility=m.get("plausibility", "medium"),
            source_chunk_ids=result.get("source_chunk_ids_used", []),
        )
        all_manifestations.append(dm)
        manif_ids.append(dm.id)
        print(f"  [{dm.plausibility:6s}] {dm.label}")

    manifestation_map[driver.id] = manif_ids

print(f"\n{'='*60}")
print(f"Total: {len(all_manifestations)} manifestations for {len(drivers)} drivers")
print(f"Average: {len(all_manifestations)/len(drivers):.1f} per driver")

## Validation: Embedding Similarity Check

Flag manifestations of the same driver that are too similar (cosine > 0.9) — they may not be truly mutually exclusive.

In [ ]:
import numpy as np
from src.llm import embed

manif_lookup = {m.id: m for m in all_manifestations}
warnings = 0

for driver in drivers:
    m_ids = manifestation_map[driver.id]
    if len(m_ids) < 2:
        print(f"⚠ {driver.name[:40]}: only {len(m_ids)} manifestation(s)")
        warnings += 1
        continue

    texts = [f"{manif_lookup[mid].label}: {manif_lookup[mid].description}" for mid in m_ids]
    embs = np.array(embed(texts))
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    sim = (embs @ embs.T) / (norms @ norms.T + 1e-10)

    for a in range(len(m_ids)):
        for b in range(a + 1, len(m_ids)):
            if sim[a][b] > 0.90:
                print(f"⚠ {driver.name[:40]}: '{manif_lookup[m_ids[a]].label}' ↔ '{manif_lookup[m_ids[b]].label}' similarity={sim[a][b]:.3f}")
                warnings += 1

if warnings == 0:
    print("✓ All manifestations pass similarity check")

## Morphological Box (Zwicky Box)

In [ ]:
driver_lookup = {d.id: d for d in drivers}

print(f"{'DRIVER':<45} | MANIFESTATIONS (optimistic → pessimistic)")
print("=" * 120)
for driver in drivers:
    m_ids = manifestation_map[driver.id]
    labels = [manif_lookup[mid].label for mid in m_ids]
    plaus = [manif_lookup[mid].plausibility for mid in m_ids]
    row = " | ".join([f"{l} [{p}]" for l, p in zip(labels, plaus)])
    print(f"{driver.name[:44]:<45} | {row}")

In [ ]:
morph_box = MorphologicalBox(
    drivers=[d.id for d in drivers],
    manifestations=manifestation_map,
    all_manifestations=all_manifestations,
)

morphbox_state = {
    "drivers": morph_box.drivers,
    "manifestations": morph_box.manifestations,
    "all_manifestations": [m.model_dump(mode="json") for m in morph_box.all_manifestations],
}

with open("../data/outputs/morphbox_state.json", "w") as f:
    json.dump(morphbox_state, f, indent=2)

print(f"Saved morphological box: {len(morph_box.drivers)} drivers × {len(morph_box.all_manifestations)} manifestations")